In [6]:
# class SparkSession in module pyspark.sql.session:
# entry point to programming Spark with the Dataset and DataFrame API.
# A SparkSession can be used to create :class:`DataFrame`, register :class:`DataFrame` as tables, execute SQL over tables, cache tables, and read parquet files.
from pyspark.sql import SparkSession

# To create a :class:`SparkSession`, use the following builder pattern:
spark = (SparkSession.builder
                .master("local")
                .appName("readingDataFromDifferentSources")
                .getOrCreate())


In [7]:
spark.range(5).show()

+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
+---+



In [12]:
# creates a :class:`DataFrame` from an :class:`RDD`, a list, a :class:`pandas.DataFrame`or a :class:`numpy.ndarray`
# data : an RDD of any kind of SQL data representation (:class:`Row`,
#  |          :class:`tuple`, ``int``, ``boolean``, etc.), or :class:`list`,
#  |          :class:`pandas.DataFrame` or :class:`numpy.ndarray`.
# When ``schema`` is a list of column names, the type of each column will be inferred from ``data``.
# When ``schema`` is ``None``, it will try to infer the schema (column names and types) from ``data``, which should be an RDD of either :class:`Row`,
# :class:`namedtuple`, or :class:`dict`
# When ``schema`` is :class:`pyspark.sql.types.DataType` or a datatype string, it must match the real data, or an exception will be thrown at runtime.
# spark.createDataFrame(data = data, schema = schema)

# Create a DataFrame from a list of tuples.
spark.createDataFrame(data = [('Shishir', 1), ('Raj', 2)]).show()

+-------+---+
|     _1| _2|
+-------+---+
|Shishir|  1|
|    Raj|  2|
+-------+---+



In [13]:
# Create a DataFrame from a list of dictionaries.
spark.createDataFrame(data = [{'name':'Shishir', 'age':23}, {'name':'Rahul', 'age':23}]).show()

+---+-------+
|age|   name|
+---+-------+
| 23|Shishir|
| 23|  Rahul|
+---+-------+



In [14]:
# Create a DataFrame with column names specified.
spark.createDataFrame(data = [('Shishir',21)], schema = ['name','age']).show()

+-------+---+
|   name|age|
+-------+---+
|Shishir| 21|
+-------+---+



In [19]:
#   Create a DataFrame with the explicit schema specified.
from pyspark.sql.types import *
schema = StructType([StructField('name',StringType()), StructField('age',IntegerType())])
df = spark.createDataFrame(data = [('Shishir',21)], schema = schema)
df.show()
display(df)
df.printSchema()

+-------+---+
|   name|age|
+-------+---+
|Shishir| 21|
+-------+---+



DataFrame[name: string, age: int]

root
 |-- name: string (nullable = true)
 |-- age: integer (nullable = true)



In [20]:
# Create a DataFrame with the schema in DDL formatted string.
spark.createDataFrame(data = [('Shishir',21), ('Rahul',23)], schema = "name: string, age: int").show()

+-------+---+
|   name|age|
+-------+---+
|Shishir| 21|
|  Rahul| 23|
+-------+---+



In [21]:
# create an empty df
# When initializing an empty DataFrame in PySpark, it's mandatory to specify its schema, 
# as the DataFrame lacks data from which the schema can be inferred.
spark.createDataFrame(data = [], schema = "name: string, age: int").show()

+----+---+
|name|age|
+----+---+
+----+---+



In [24]:
# Create a DataFrame from Row objects.
from pyspark.sql import Row
Person = Row('name', 'age')
df = spark.createDataFrame(data = [Person('Shishir',21), Person('Rahul',23)])
df.show()

+-------+---+
|   name|age|
+-------+---+
|Shishir| 21|
|  Rahul| 23|
+-------+---+



In [25]:
# Create a DataFrame from a pandas DataFrame.
spark.createDataFrame(df.toPandas()).show()

+-------+---+
|   name|age|
+-------+---+
|Shishir| 21|
|  Rahul| 23|
+-------+---+



In [27]:
# range() 
# Create a :class:`DataFrame` with single :class:`pyspark.sql.types.LongType` column named
# ``id``, containing elements in a range from ``start`` to ``end`` (exclusive) with
# step value ``step``
spark.range(start = 1, end = 10, step = 2).show()

+---+
| id|
+---+
|  1|
|  3|
|  5|
|  7|
|  9|
+---+



In [28]:
# if only 1arg is specified, it will be treated as end point(exlusive)
spark.range(5).show()

+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
+---+



In [33]:
# spark.sql(sqlQuery, args, kwargs) Returns a :class:`DataFrame` representing the result of the given query.
spark.sql("select * from range(10) where id > {minVal} and id < {maxVal}", minVal = 5, maxVal = 10 ).show()


+---+
| id|
+---+
|  6|
|  7|
|  8|
|  9|
+---+



In [46]:
spark.sql("""
          select t1.val1, t2.val2 
          from {tbl1} t1 inner join {tbl2} t2 on t1.key = t2.key
        """, 
        tbl1 = spark.createDataFrame(data = [(1,"A"), (2,"B")], schema = ["val1", "key"]),
        tbl2 = spark.createDataFrame(data = [(3,"A"), (4,"B"), (5,"C")], schema = ["val2", "key"])).show()

+----+----+
|val1|val2|
+----+----+
|   1|   3|
|   2|   4|
+----+----+



In [36]:
spark.createDataFrame(data = [(1,"A"), (2,"B")], schema = ["val1", "key"]).show()

+----+---+
|val1|key|
+----+---+
|   1|  A|
|   2|  B|
+----+---+



In [39]:
spark.createDataFrame(data = [(3,"A"), (4,"B"), (5,"C")], schema = ["val2", "key"]).show()

+----+---+
|val2|key|
+----+---+
|   3|  A|
|   4|  B|
|   5|  C|
+----+---+



In [57]:
# spark.table("tableName") -->  Returns the specified table as a :class:`DataFrame`.

# spark.range(5) -- returns a dataframe
# df.createOrReplaceTempView("table1")-- registers the df as table table1
spark.range(5).createOrReplaceTempView("table1")
spark.table("table1").sort("id").show()

+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
+---+



In [61]:
# spark.catalog --  Interface through which the user may create, drop, alter or query underlying databases, tables, functions, etc.
spark.range(5).createOrReplaceTempView("test_view")
spark.catalog.listTables()

[Table(name='table1', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True),
 Table(name='test_view', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True)]

In [62]:
spark.catalog.dropTempView("test_view")

True

In [ ]:
# Creates a :class:`Builder` for constructing a :class:`SparkSession`.
# SparkSession.builder

In [75]:
# conf
#  Runtime configuration interface for Spark.
#       
#  This is the interface through which the user can get and set all Spark and Hadoop
#  configurations that are relevant to Spark SQL. When getting the value of a config,
#  this defaults to the value set in the underlying :class:`SparkContext`, if any.

spark.conf.set("key", "value")
spark.conf.get("key")

-------------------------------------------
Batch: 296
-------------------------------------------
+--------------------+-----+
|           timestamp|value|
+--------------------+-----+
|2026-01-14 14:52:...|  295|
+--------------------+-----+



'value'

-------------------------------------------
Batch: 297
-------------------------------------------
+--------------------+-----+
|           timestamp|value|
+--------------------+-----+
|2026-01-14 14:52:...|  296|
+--------------------+-----+

-------------------------------------------
Batch: 298
-------------------------------------------
+--------------------+-----+
|           timestamp|value|
+--------------------+-----+
|2026-01-14 14:52:...|  297|
+--------------------+-----+

-------------------------------------------
Batch: 299
-------------------------------------------
+--------------------+-----+
|           timestamp|value|
+--------------------+-----+
|2026-01-14 14:52:...|  298|
+--------------------+-----+

-------------------------------------------
Batch: 300
-------------------------------------------
+--------------------+-----+
|           timestamp|value|
+--------------------+-----+
|2026-01-14 14:52:...|  299|
+--------------------+-----+

--------------------

In [65]:
#  read
#  |      Returns a :class:`DataFrameReader` that can be used to read data
#  |      in as a :class:`DataFrame`.

# Write a DataFrame into a JSON file and read it back.
spark.range(5).write.mode("overwrite").format("json").save("/Users/shishir/Pyspark/SampleData")
spark.read.json("/Users/shishir/Pyspark/SampleData").show()

+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
+---+



In [77]:
# readStream 
#     Returns a :class:`DataStreamReader` that can be used to read data streams
#     as a streaming :class:`DataFrame`.

# The example below uses Rate source that generates rows continuously.
# and then write the stream out to the console.
# The streaming query stops in 3 seconds.

import time


# Commenting this code bcoz it keeps on writing output to the console for some reason
# df = spark.readStream.format("rate").load()

# q = df.writeStream.format("console").start()

# time.sleep(3)

# q.stop()


-------------------------------------------
Batch: 353
-------------------------------------------
+--------------------+-----+
|           timestamp|value|
+--------------------+-----+
|2026-01-14 14:53:...|  352|
+--------------------+-----+

-------------------------------------------
Batch: 354
-------------------------------------------
+--------------------+-----+
|           timestamp|value|
+--------------------+-----+
|2026-01-14 14:53:...|  353|
+--------------------+-----+

-------------------------------------------
Batch: 355
-------------------------------------------
+--------------------+-----+
|           timestamp|value|
+--------------------+-----+
|2026-01-14 14:53:...|  354|
+--------------------+-----+

-------------------------------------------
Batch: 356
-------------------------------------------
+--------------------+-----+
|           timestamp|value|
+--------------------+-----+
|2026-01-14 14:53:...|  355|
+--------------------+-----+

--------------------

In [80]:
spark.stop()

In [86]:
# sparkContext
#  |      Returns the underlying :class:`SparkContext`.


spark = (SparkSession.builder
        .master("local")
        .getOrCreate())

# Create an RDD from the Spark context
rdd = spark.sparkContext.parallelize([1, 2, 3])
rdd.collect()



[1, 2, 3]

In [92]:
sq = (spark.readStream.format("rate").load()
                    .writeStream.format("memory").queryName('this_query').start())
sqm = spark.streams  # returns instance of pyspark.sql.streaming.query.StreamingQueryManager

sq.stop()

26/01/14 14:59:25 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /private/var/folders/tr/k980qy9n2kl4d_dmml8c0k780000gn/T/temporary-e542f4c7-7921-4cf5-892f-88b6f6e57940. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/01/14 14:59:25 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


IllegalArgumentException: Cannot start query with name this_query as a query with that name is already active in this SparkSession

In [94]:
# udf
#  |      Returns a :class:`UDFRegistration` for UDF registration.

#  Register a Python UDF, and use it in SQL.
spark.udf.register("strlen", lambda x: len(x))
spark.sql("select strlen('test')").show()

+------------+
|strlen(test)|
+------------+
|           4|
+------------+



In [97]:
help(spark.udf)

Help on UDFRegistration in module pyspark.sql.udf object:

class UDFRegistration(builtins.object)
 |  UDFRegistration(sparkSession: 'SparkSession')
 |  
 |  Wrapper for user-defined function registration. This instance can be accessed by
 |  :attr:`spark.udf` or :attr:`sqlContext.udf`.
 |  
 |  .. versionadded:: 1.3.1
 |  
 |  Methods defined here:
 |  
 |  __init__(self, sparkSession: 'SparkSession')
 |      Initialize self.  See help(type(self)) for accurate signature.
 |  
 |  register(self, name: str, f: Union[Callable[..., Any], ForwardRef('UserDefinedFunctionLike')], returnType: Optional[ForwardRef('DataTypeOrString')] = None) -> 'UserDefinedFunctionLike'
 |      Register a Python function (including lambda function) or a user-defined function
 |      as a SQL function.
 |      
 |      .. versionadded:: 1.3.1
 |      
 |      .. versionchanged:: 3.4.0
 |          Supports Spark Connect.
 |      
 |      Parameters
 |      ----------
 |      name : str,
 |          name of the us

In [4]:
help(SparkSession)

Help on class SparkSession in module pyspark.sql.session:

class SparkSession(pyspark.sql.pandas.conversion.SparkConversionMixin)
 |  SparkSession(sparkContext: pyspark.context.SparkContext, jsparkSession: Optional[py4j.java_gateway.JavaObject] = None, options: Dict[str, Any] = {})
 |  
 |  The entry point to programming Spark with the Dataset and DataFrame API.
 |  
 |  A SparkSession can be used to create :class:`DataFrame`, register :class:`DataFrame` as
 |  tables, execute SQL over tables, cache tables, and read parquet files.
 |  To create a :class:`SparkSession`, use the following builder pattern:
 |  
 |  .. versionchanged:: 3.4.0
 |      Supports Spark Connect.
 |  
 |  .. autoattribute:: builder
 |     :annotation:
 |  
 |  Examples
 |  --------
 |  Create a Spark session.
 |  
 |  >>> spark = (
 |  ...     SparkSession.builder
 |  ...         .master("local")
 |  ...         .appName("Word Count")
 |  ...         .config("spark.some.config.option", "some-value")
 |  ...      

26/01/14 12:52:09 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors
